In [1]:
%%capture

%pip install -q \
    "ragas>=0.2.15" \
    "langchain-community<0.4.2" \
    langchain_core \
    langchain_huggingface \
    datasets pandas matplotlib seaborn

In [2]:
import os
import asyncio
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1,      # low temperature for deterministic judging
    do_sample=False,
    return_full_text=False,
)

hf_llm = HuggingFacePipeline(pipeline=pipe)
judge_llm = LangchainLLMWrapper(hf_llm)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_14827/1254010449.py:22: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(hf_llm)


In [4]:
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-8B"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_ID,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
judge_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/tmp/ipykernel_14827/3312741499.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)


In [5]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train")

df = ds.to_pandas()

In [6]:
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [7]:
from ragas.run_config import RunConfig

run_config = RunConfig(timeout=1800, max_workers=2, max_retries=3)

In [8]:
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness

metrics = [
    SummarizationScore(llm=judge_llm, coeff=0.5),  # 0.5 balances QA vs. conciseness
    SemanticSimilarity(embeddings=judge_embeddings),
    # AnswerCorrectness(llm=judge_llm, embeddings=judge_embeddings)
]

/tmp/ipykernel_14827/3542372934.py:1: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_14827/3542372934.py:1: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_14827/3542372934.py:1: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summari

In [9]:
base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=run_config,
)

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warni

In [10]:
base_result

{'summary_score': 0.4676, 'semantic_similarity': 0.6978}

In [11]:
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=run_config,
)

ft_result

Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'summary_score': 0.4782, 'semantic_similarity': 0.6720}